In [1]:
import os
os.chdir('../../../../..')

In [2]:
import numpy as np

from sklearn.cluster import AgglomerativeClustering, SpectralClustering, DBSCAN
from kmedoids import KMedoids

from src.datasets import QM9Dataset
from src.helper_functions import plot_molecules_with_py3dmol, create_chemiscope_viewer, plot_distance_matrix_projection, evaluate_distance_matrix_clustering_sweep, average_numeric_by_cluster

In [3]:
qm9 = QM9Dataset(limit=5000, sampling_strategy="stratified", stratify_by=["num_atoms", "gap"], add_acsf=True)
df = qm9.load()
molecules = qm9.get_molecules()

2026-04-21 10:37:24.418 | INFO     | src.datasets:load:496 - Loading cached full QM9 dataset from: data/QM9/dataset_cleaned.parquet
2026-04-21 10:37:24.919 | INFO     | src.datasets:_sample_qm9_df:688 - QM9 sampling complete: strategy=stratified, requested_limit=5500, returned_rows=5500.
2026-04-21 10:37:24.921 | INFO     | src.datasets:_add_requested_descriptors:129 - Applying requested QM9 descriptors to sampled dataframe (rows=5500).
2026-04-21 10:37:24.922 | INFO     | src.features:compute_acsf:201 - Computing ACSF (rcut=6.0)...
2026-04-21 10:39:16.678 | SUCCESS  | src.datasets:add_acsf:853 - Added ACSF embeddings.
2026-04-21 10:39:16.686 | INFO     | src.datasets:_add_requested_descriptors:152 - Added descriptor column(s): ['acsf_embedding']
2026-04-21 10:39:16.745 | INFO     | src.datasets:_drop_rows_with_null_required_descriptors:581 - Dropped QM9 rows with null/empty descriptor vectors: dropped=23, remaining=5477, descriptor_cols=['acsf_embedding'].
2026-04-21 10:39:16.747 | IN

In [4]:
len(molecules[0:2])

2

In [5]:
plot_molecules_with_py3dmol(molecules[0:3])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

In [5]:
dist_matrix = qm9.get_distance_matrix(
    descriptor="acsf",
    dist_type="euclidean",
    force_calculate=True,
    pca_variance=99
)

2026-04-21 10:40:41.367 | INFO     | src.datasets:get_distance_matrix:991 - Applying PCA to retain 99.00% of variance.
2026-04-21 10:40:41.734 | INFO     | src.datasets:get_distance_matrix:1001 - PCA reduced 'acsf' dimensions from 65 to 5
2026-04-21 10:40:41.838 | INFO     | src.datasets:get_distance_matrix:1012 - Calculating distance matrix for acsf using euclidean distance.
2026-04-21 10:40:42.393 | SUCCESS  | src.distance:_compute_and_save:79 - Saved distance matrix to data/QM9/dist_acsf_euclidean_pca0.99.npy


# Determining the best number of clusters for each clustering method

In [6]:
out = evaluate_distance_matrix_clustering_sweep(
    dist_matrix=dist_matrix,
    fingerprint="acsf",
    distance_metric="euclidean",
    dataset_name="qm9",
)

Evaluation k:   0%|          | 0/19 [00:05<?, ?it/s]


KeyboardInterrupt: 

In [7]:
# find the n molecules that are not on the diagonal with the smallest distance
n = 10
# Get the indices of the upper triangle (excluding diagonal)
triu_indices = np.triu_indices_from(dist_matrix, k=1)
# Get the distances and corresponding molecule pairs
distances = dist_matrix[triu_indices]
molecule_pairs = list(zip(triu_indices[0], triu_indices[1]))
# Get the indices of the n smallest distances
smallest_indices = np.argsort(distances)[:n]
# Get the corresponding molecule pairs for the n smallest distances
closest_pairs = [molecule_pairs[i] for i in smallest_indices]
print("Closest molecule pairs (indices):", closest_pairs)
mols = [(molecules[idx1], molecules[idx2]) for idx1, idx2 in closest_pairs]

Closest molecule pairs (indices): [(np.int64(1332), np.int64(1333)), (np.int64(3300), np.int64(3313)), (np.int64(2814), np.int64(2845)), (np.int64(1567), np.int64(4667)), (np.int64(1429), np.int64(1430)), (np.int64(2849), np.int64(2957)), (np.int64(2756), np.int64(3460)), (np.int64(3389), np.int64(3698)), (np.int64(3244), np.int64(3413)), (np.int64(1473), np.int64(1488))]


In [8]:
print(mols[0])

(Atoms(symbols='H3CH3NHOCNCNC2', pbc=False, initial_charges=..., mass=..., partial_charge=...), Atoms(symbols='H3CH3NHOCNCNC2', pbc=False, initial_charges=..., mass=..., partial_charge=...))


In [9]:
plot_molecules_with_py3dmol(mols[1])

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

# Hiercical Clustering on Distance Matrix

In [10]:
model_hier = AgglomerativeClustering(metric='precomputed', n_clusters=3, linkage='complete')
labels_hier = model_hier.fit_predict(dist_matrix)
df = df.with_columns(labels_hier=labels_hier)

In [15]:
create_chemiscope_viewer(df, dist_matrix, labels_hier, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [13]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="acsf",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_hier,
    clustering_method="hierarchical"
)

2026-04-21 10:46:23.679 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/qm9/clustering/euclidean/acsf/pca_hierarchical_projection.png


{'coords': array([[-43.62463075,  26.47490526],
        [ 72.80547255,   0.14337807],
        [ 99.82023045,  33.12011067],
        ...,
        [-33.22654004,  41.25085348],
        [-45.36688247,  39.99436312],
        [-31.06818226, -25.73946776]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/qm9/clustering/euclidean/acsf/pca_hierarchical_projection.png'),
 'output_dir': PosixPath('figures/qm9/clustering/euclidean/acsf'),
 'clustering_method': 'hierarchical'}

In [14]:
average_numeric_by_cluster(df, "labels_hier")

shape: (3, 59)
┌─────────────┬───────┬─────────────────────┬───────────┬────────────┬─────────┬─────────┬───────────────────┬─────────────────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────────┬─────────────────┬────────────────┬─────────────────┬─────────────────┬───────────────────┬─────────────────┬─────────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬────────────────────┬──────────┬───────────┬──────────┬──────────┬────────────┬────────┬─────────┬─────────┬─────────┬────────┬───────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬──────────┬──────────┬──────────┬──────────┬────────┬────────┬────────┬────────────────────┬──────────────┬─────────────┐
│ labels_hier ┆ count ┆ token_to_atom_ratio ┆ num_atoms ┆ mol_weight ┆ logp    ┆ tpsa    ┆ election_affinity ┆ ionization_energies ┆ num_heavy_atoms ┆ num_ring

labels_hier,count,token_to_atom_ratio,num_atoms,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,pct_aliphatic_ring,pct_aromatic,pct_acyclic
i64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,2684,2.006618,16.52608,121.77608,-0.086811,43.917288,0.85701,12.835974,8.732116,1.510805,0.199329,2.050815,2.042474,0.099473,0.297548,0.60298,0.972057,2.374441,5.943741,0.636736,1.65611,3.627049,6.334203,33.308867,1.264695,0.002235,0.331967,0.043592,0.168778,0.188152,0.00149,0.066319,0.168405,0.601714,0.000373,2.812221,3.004526,72.246434,-6.556377,-0.13877,6.417587,1166.563846,3.584578,-11393.184992,-11392.958081,-11392.93238,-11394.093941,30.573018,-71.159627,-71.569661,-71.968695,-66.319196,3.608075,1.400782,1.10543,67.362146,19.113264,13.52459
1,2290,2.213586,20.327948,123.685153,0.291266,25.757205,0.905336,12.82896,8.823144,1.874672,0.005677,2.075538,2.656332,0.031875,0.079735,0.88839,0.847598,1.434934,7.265502,0.229258,0.567249,6.258079,6.349345,44.965939,1.259107,0.000437,0.405677,0.0,0.095197,0.053275,0.001747,0.012227,0.082533,0.519651,0.0,1.768559,2.18709,79.275991,-6.513554,1.003652,7.517228,1212.349888,4.781825,-10790.169492,-10789.929608,-10789.903925,-10791.08262,33.523398,-83.875561,-84.419138,-84.915934,-77.94025,3.077594,1.398738,1.15437,91.572052,0.567686,7.860262
2,26,1.615254,10.653846,107.115385,-0.307692,66.769231,0.165998,13.366187,7.692308,1.076923,1.0,2.013403,0.807692,0.144231,0.759615,0.096154,0.769231,4.192308,3.576923,0.5,2.076923,0.269231,5.576923,17.269231,1.271154,0.0,0.0,0.115385,0.346154,0.038462,0.0,0.0,0.0,0.038462,0.0,4.846154,3.727869,55.503846,-7.209866,-1.609763,5.599999,783.654543,1.938422,-10760.73937,-10760.572284,-10760.54665,-10761.569167,21.892923,-46.660842,-46.904404,-47.152457,-43.525034,6.497415,2.018632,1.506122,7.692308,92.307692,0.0


# KMedoids

In [ ]:
model_km = KMedoids(n_clusters=3, metric="precomputed")
labels_km = model_km.fit_predict(dist_matrix)
df = df.with_columns(labels_km=labels_km)
print(np.unique(labels_km, return_counts=True))

In [17]:
create_chemiscope_viewer(df, dist_matrix, labels_km, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [18]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="acsf",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_km,
    clustering_method="kmedoids"
)

2026-04-21 10:53:07.118 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/qm9/clustering/euclidean/acsf/pca_kmedoids_projection.png


{'coords': array([[-43.62463075,  26.47490526],
        [ 72.80547255,   0.14337807],
        [ 99.82023045,  33.12011067],
        ...,
        [-33.22654004,  41.25085348],
        [-45.36688247,  39.99436312],
        [-31.06818226, -25.73946776]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/qm9/clustering/euclidean/acsf/pca_kmedoids_projection.png'),
 'output_dir': PosixPath('figures/qm9/clustering/euclidean/acsf'),
 'clustering_method': 'kmedoids'}

In [19]:
average_numeric_by_cluster(df, "labels_km")

shape: (3, 60)
┌───────────┬───────┬─────────────────────┬───────────┬────────────┬─────────┬─────────┬───────────────────┬─────────────────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────────┬─────────────────┬────────────────┬─────────────────┬─────────────────┬───────────────────┬─────────────────┬─────────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬────────────────────┬──────────┬───────────┬──────────┬──────────┬────────────┬────────┬─────────┬─────────┬─────────┬────────┬───────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬──────────┬──────────┬──────────┬──────────┬────────┬────────┬────────┬─────────────┬────────────────────┬──────────────┬─────────────┐
│ labels_km ┆ count ┆ token_to_atom_ratio ┆ num_atoms ┆ mol_weight ┆ logp    ┆ tpsa    ┆ election_affinity ┆ ionization_energies ┆ num_heavy_atoms 

labels_km,count,token_to_atom_ratio,num_atoms,mol_weight,logp,tpsa,election_affinity,ionization_energies,num_heavy_atoms,num_rings,num_aromatic_rings,coordination,num_rotatable_bonds,fraction_csp1,fraction_csp2,fraction_csp3,h_bond_donors,h_bond_acceptors,branching_index,num_sp_carbons,num_sp2_carbons,num_sp3_carbons,main_chain_length,raw_token_count,avg_bond_length,fr_benzene,fr_alcohol,fr_phenol,fr_amine,fr_amide,fr_carboxylic_acid,fr_ester,fr_ketone,fr_ether,fr_nitro,fr_halogen,mu,alpha,homo,lumo,gap,r2,zpve,u0,u,h,g,cv,u0_atom,u_atom,h_atom,g_atom,A,B,C,labels_hier,pct_aliphatic_ring,pct_aromatic,pct_acyclic
u64,u32,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
0,1922,2.101834,17.719563,123.221644,-0.101457,38.193548,0.901509,12.819029,8.808533,1.78616,0.037981,2.077917,2.223205,0.076416,0.162253,0.761331,0.931322,2.14204,6.319979,0.521332,1.027575,4.769511,6.291363,37.209677,1.266346,0.00052,0.429761,0.005203,0.111342,0.12539,0.002081,0.05411,0.162851,0.710718,0.0,2.490114,2.853914,73.827758,-6.613189,0.191359,6.804524,1169.034313,3.947662,-11334.860747,-11334.630284,-11334.604588,-11335.770314,31.511982,-75.063946,-75.516424,-75.946146,-69.874297,3.285536,1.402536,1.139937,0.208117,87.044745,3.798127,9.157128
1,1923,2.225683,20.700468,123.782111,0.370255,23.331253,0.912465,12.823629,8.830993,1.906396,0.00364,2.07698,2.694228,0.024953,0.072714,0.902333,0.796152,1.320333,7.404056,0.185647,0.529901,6.472699,6.358294,46.024441,1.258666,0.00052,0.394696,0.0,0.076443,0.042642,0.00104,0.00988,0.076963,0.50442,0.0,1.642746,2.074735,80.11168,-6.498158,1.114839,7.613013,1221.030462,4.899701,-10718.565653,-10718.324671,-10718.298988,-10719.479438,33.774623,-85.207482,-85.764314,-86.270688,-79.165566,3.057525,1.395538,1.154018,0.982839,91.679667,0.364015,7.956318
2,1155,1.884983,14.995671,119.485714,-0.078788,52.225108,0.770892,12.882756,8.597403,1.105628,0.427706,2.010328,1.845887,0.128894,0.475569,0.395537,1.081385,2.694372,5.45368,0.768831,2.427706,2.129004,6.378355,28.398268,1.261051,0.004329,0.203463,0.095238,0.27619,0.264069,0.001732,0.071861,0.155844,0.406926,0.000866,3.271861,3.198761,70.080381,-6.488573,-0.543356,5.945228,1153.929087,3.127495,-11403.611924,-11403.389976,-11403.364273,-11404.518298,29.334327,-65.933995,-66.290016,-66.6497,-61.542652,4.074699,1.416452,1.073165,0.045022,40.779221,40.692641,18.528139


# Spectral

In [20]:
model_spectral = SpectralClustering(
                n_clusters=3,
                affinity="precomputed",
                assign_labels='kmeans',
                random_state=42,
            )

labels_spectral = model_spectral.fit_predict(dist_matrix)
df = df.with_columns(labels_spectral=labels_spectral)
print(np.unique(labels_spectral, return_counts=True))

(array([0, 1, 2], dtype=int32), array([4998,    1,    1]))


In [21]:
create_chemiscope_viewer(df, dist_matrix, labels_spectral, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [21]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="acsf",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_spectral,
    clustering_method="spectral"
)

2026-04-19 12:29:48.193 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/qm9/clustering/euclidean/acsf/pca_spectral_projection.png


{'coords': array([[-42.87330632,  26.97616543],
        [ 72.57577087,   0.38978408],
        [ 99.66756854,  32.53057816],
        ...,
        [-33.04061161,  41.09393408],
        [-45.10445665,  39.92296242],
        [-31.10718847, -25.59146542]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/qm9/clustering/euclidean/acsf/pca_spectral_projection.png'),
 'output_dir': PosixPath('figures/qm9/clustering/euclidean/acsf'),
 'clustering_method': 'spectral'}

In [22]:
average_numeric_by_cluster(df, "labels_spectral")

shape: (4, 61)
┌─────────────────┬───────┬─────────────────────┬───────────┬────────────┬────────┬─────────┬───────────────────┬─────────────────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────────┬─────────────────┬────────────────┬─────────────────┬─────────────────┬───────────────────┬─────────────────┬─────────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬────────────────────┬──────────┬───────────┬──────────┬──────────┬────────────┬────────┬─────────┬─────────┬─────────┬────────┬───────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬──────────┬──────────┬──────────┬──────────┬────────┬────────┬────────┬─────────────┬───────────┬────────────────────┬──────────────┬─────────────┐
│ labels_spectral ┆ count ┆ token_to_atom_ratio ┆ num_atoms ┆ mol_weight ┆ logp   ┆ tpsa    ┆ election_affinity ┆ ionization_energ

# DBSCAN 

In [22]:
model_db = DBSCAN(
    eps=0.6,
    min_samples=2,
    metric='precomputed',
)

labels_db = model_db.fit_predict(dist_matrix)
df = df.with_columns(labels_db=labels_db)
print(np.unique(labels_db, return_counts=True))

(array([-1,  0,  1,  2,  3,  4,  5]), array([  19, 4970,    2,    3,    2,    2,    2]))


In [23]:
create_chemiscope_viewer(df, dist_matrix, labels_db, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [24]:
plot_distance_matrix_projection(
    dist_matrix=dist_matrix,
    fingerprint="acsf",
    distance_metric="euclidean",
    projection_method="PCA",
    dataset_name="qm9",
    labels=labels_db,
    clustering_method="dbscan"
)

2026-04-21 11:08:34.778 | SUCCESS  | src.helper_functions:plot_distance_matrix_projection:385 - Saved PCA projection plot to figures/qm9/clustering/euclidean/acsf/pca_dbscan_projection.png


{'coords': array([[-43.62463075,  26.47490526],
        [ 72.80547255,   0.14337807],
        [ 99.82023045,  33.12011067],
        ...,
        [-33.22654004,  41.25085348],
        [-45.36688247,  39.99436312],
        [-31.06818226, -25.73946776]], shape=(5000, 2)),
 'figure_path': PosixPath('figures/qm9/clustering/euclidean/acsf/pca_dbscan_projection.png'),
 'output_dir': PosixPath('figures/qm9/clustering/euclidean/acsf'),
 'clustering_method': 'dbscan'}

In [26]:
average_numeric_by_cluster(df, "labels_db")

shape: (12, 62)
┌───────────┬───────┬─────────────────────┬───────────┬────────────┬─────────┬─────────┬───────────────────┬─────────────────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────────┬─────────────────┬────────────────┬─────────────────┬─────────────────┬───────────────────┬─────────────────┬─────────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬────────────────────┬──────────┬───────────┬──────────┬──────────┬────────────┬────────┬─────────┬─────────┬─────────┬─────────┬───────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬───────────┬───────────┬───────────┬──────────┬─────────┬────────┬────────┬─────────────┬───────────┬─────────────────┬────────────────────┬──────────────┬─────────────┐
│ labels_db ┆ count ┆ token_to_atom_ratio ┆ num_atoms ┆ mol_weight ┆ logp    ┆ tpsa    ┆ election_affinity ┆ io

# KMeans

In [27]:
from sklearn.cluster import KMeans
kmeans = KMeans(n_clusters=3)
X = np.array(df['acsf_embedding'].to_list())
labels_kmeans = kmeans.fit_predict(X)
df = df.with_columns(labels_kmeans=labels_kmeans)
print(np.unique(labels_kmeans, return_counts=True))


(array([0, 1, 2], dtype=int32), array([1818, 1038, 2144]))


In [29]:
create_chemiscope_viewer(df, X, labels_kmeans, 'PCA')

Running PCA dimensionality reduction...
Converting structures/molecules to ASE Atoms for Chemiscope...
Assembling properties for Chemiscope...
Generating Chemiscope widget...
Saved Chemiscope input to: qm9_PCA_clustering.json
If the viewer does not open automatically, run `chemiscope show qm9_PCA_clustering.json`.


<ChemiscopeWidget(meta={'name': 'QM9 - PCA Clustering'}, settings={'map': {'x': {'property': 'PCA_1'}, 'y': {'…

In [30]:
average_numeric_by_cluster(df, 'labels_kmeans')

shape: (3, 63)
┌───────────────┬───────┬─────────────────────┬───────────┬────────────┬─────────┬─────────┬───────────────────┬─────────────────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────────────┬───────────────┬───────────────┬───────────────┬───────────────┬──────────────────┬─────────────────┬────────────────┬─────────────────┬─────────────────┬───────────────────┬─────────────────┬─────────────────┬────────────┬────────────┬───────────┬──────────┬──────────┬────────────────────┬──────────┬───────────┬──────────┬──────────┬────────────┬────────┬─────────┬─────────┬─────────┬────────┬───────────┬────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────┬──────────┬──────────┬──────────┬──────────┬────────┬────────┬────────┬─────────────┬───────────┬─────────────────┬───────────┬────────────────────┬──────────────┬─────────────┐
│ labels_kmeans ┆ count ┆ token_to_atom_ratio ┆ num_atoms ┆ mol_weight ┆ logp    ┆ tpsa    ┆ election